In [22]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import export_text
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestClassifier
!pip install xgboost
import xgboost as xgb


data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-06-trees/CreditScoring.csv'

df = pd.read_csv(data)

df.columns = df.columns.str.lower()

status_values = {
    1: 'ok',
    2: 'default',
    0: 'unk'
}

df.status = df.status.map(status_values)

home_values = {
    1: 'rent',
    2: 'owner',
    3: 'private',
    4: 'ignore',
    5: 'parents',
    6: 'other',
    0: 'unk'
}

df.home = df.home.map(home_values)

marital_values = {
    1: 'single',
    2: 'married',
    3: 'widow',
    4: 'separated',
    5: 'divorced',
    0: 'unk'
}

df.marital = df.marital.map(marital_values)

records_values = {
    1: 'no',
    2: 'yes',
    0: 'unk'
}

df.records = df.records.map(records_values)

job_values = {
    1: 'fixed',
    2: 'partime',
    3: 'freelance',
    4: 'others',
    0: 'unk'
}

df.job = df.job.map(job_values)

df.describe().round()

for c in ['income', 'assets', 'debt']:
    df[c] = df[c].replace(to_replace=99999999, value=np.nan)
    
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=11)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=11)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = (df_train.status == "default").astype("int").values
y_val = (df_val.status == "default").astype("int").values
y_test = (df_test.status == "default").astype("int").values

del df_train['status']
del df_val['status']
del df_test['status']

train_dicts = df_train.fillna(0).to_dict(orient='records')
val_dict = df_val.fillna(0).to_dict(orient = 'records')

dv = DictVectorizer(sparse=False)

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dict)

max_auc = 0
d1=0
l1=0
e1=0

estimetors = [50, 75, 100]
depth = [2, 4, 5, 7, 10]
leafs = [20, 30]

for e in estimetors:
    for d in depth:
        for l in leafs:
            rf = RandomForestClassifier(n_estimators=e, max_depth=d, max_leaf_nodes=l)
            rf.fit(X_train, y_train)

            y_pred = rf.predict_proba(X_val)[: , 1]
            cur_auc = roc_auc_score(y_val, y_pred)
            if cur_auc > max_auc:
                max_auc = cur_auc
                d1=d
                l1=l
                e1=e
print(max_auc, e1, d1, l1) # Найкращі параметри dt (d = 9, l = 18) # Найкращі параметри rf (e = 100, d = 10, l = 30)


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
0.8359126812445148 100 10 30


In [18]:

dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

max_auc = 0
best_params = {}

etas = [0.05, 0.1, 0.3]
depths = [3, 4, 6]
weights = [1, 10, 30]

for e in etas:
    for d in depths:
        for w in weights:
            
            xgb_params = {
                'eta': e,
                'max_depth': d,
                'min_child_weight': w,
                'objective': 'binary:logistic',
                'verbosity': 0,
                'seed': 1
            }
            
            model = xgb.train(xgb_params, dtrain, num_boost_round=100)
            
            y_pred = model.predict(dval)
            
            auc = roc_auc_score(y_val, y_pred)
            
            if auc > max_auc:
                max_auc = auc
                best_params = {'eta': e, 'max_depth': d, 'min_child_weight': w}

print(f"Найкращий AUC: {round(max_auc, 4)}")
print(f"Ідеальні параметри: {best_params}") # Ідеальні параметри: {'eta': 0.3, 'max_depth': 3, 'min_child_weight': 30}

Найкращий AUC: 0.8491
Ідеальні параметри: {'eta': 0.3, 'max_depth': 3, 'min_child_weight': 30}


In [23]:
df_full_train = df_full_train.reset_index(drop=True)

y_full_train = (df_full_train.status == 'default').astype(int).values
del df_full_train['status']

full_train_dict = df_full_train.to_dict(orient="records")
test_dict = df_test.to_dict(orient="records")

dv = DictVectorizer(sparse=False)

X_full_train = dv.fit_transform(full_train_dict)
X_test = dv.transform(test_dict)

d_full_train = xgb.DMatrix(data = X_full_train, label=y_full_train)
d_test = xgb.DMatrix(data = X_test, label=y_test)

xgb_params = {
    'eta': 0.3,
    'max_depth': 3,
    'min_child_weight': 30,

    'objective': 'binary:logistic',
    'eval_metric': 'auc',

    'nthread': 8,
    'seed': 1,
    'verbosity': 0,
}
model =xgb.train(xgb_params, d_full_train, 200)

y_pred3 = model.predict(d_test)
roc_auc_score(y_test, y_pred3)

0.8286197540013626